# Document Intelligence with LLMs

A huge share of real-world "give an LLM access to our data" work comes down to one unglamorous problem: getting messy documents into a form a model can reason about. PDFs are the worst offenders — a single file can mix clean paragraphs, dense tables, and pages that are really just a photo of paper.

> This assignment has you build small implementations of the three main ways people solve that problem today, on the same document, so you can see where each one shines and where it quietly fails.

By the end, you'll have called an LLM API directly, built a page-image pipeline for a multimodal model, used a model's native PDF understanding, and made an evidence-based case for which approach fits which situation.

---

## Learning objectives

* **Use an LLM API directly:** authentication, request/response structure, and the parameters that actually matter (max_output_tokens, instructions, temperature)

* **Extract text and tables from PDFs with a traditional, non-LLM parser** — and recognize exactly where that approach breaks down

* **Send images to a multimodal LLM and get structured output back**

* **Use a model's native PDF-understanding endpoint** and compare it to the "roll your own" image pipeline

* **Reason about cost, latency, and accuracy trade-offs** well enough to justify a design choice, not just report numbers

---

## Fill in every `# TODO`

**Provided alongside this notebook:** `sample_field_report.pdf` (place it in the same folder as this notebook, or update `SAMPLE_PDF` below).





## Setup (5 pts)

In [3]:
# Run once:
!pip install openai pdfplumber pdf2image pillow pandas


Defaulting to user installation because normal site-packages is not writeable


In [20]:
import os
import base64
import io
from difflib import SequenceMatcher

import pandas as pd
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path
from openai import OpenAI

from pathlib import Path
from dotenv import load_dotenv

env_path = Path(__file__).resolve().parent / ".env" if "__file__" in globals() else Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path, override=True)

api_key = os.environ["OPENROUTER_API_KEY"]

client = OpenAI(
    api_key=api_key,
    base_url="https://api.avalai.ir/v1",
)

# Cheap/fast default; used for most calls in this notebook.
MODEL_FAST = "openai/gpt-4.1-mini"
# Stronger vision quality; use this for Part 3.
MODEL_VISION = "openai/gpt-4.1"

SAMPLE_PDF = "sample_field_report.pdf"


In [21]:
# Sanity check: confirm the sample PDF is where we expect it and how many pages it has
with pdfplumber.open(SAMPLE_PDF) as pdf:
    print(f"{SAMPLE_PDF}: {len(pdf.pages)} pages")


sample_field_report.pdf: 3 pages


## Part 1 — LLM API fundamentals (20 pts)

Implement `call_llm` below, then use it for the three exploratory calls that follow.

In [22]:
def call_llm(prompt, system=None, model=MODEL_FAST, max_output_tokens=1024):
    """Call the LLM and return its text response as a string.

    Args:
        prompt: the user message text
        system: optional system prompt, passed through as the Responses API's
            `instructions` parameter
        model: model id to use
        max_output_tokens: max tokens to generate

    Returns:
        The model's reply as a plain string.
    """
    kwargs = dict(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
    )
    if system is not None:
        kwargs["instructions"] = system

    response = client.responses.create(**kwargs)
    return response.output_text

In [23]:
# 1) A basic call, no system prompt
basic_answer = call_llm("What are the four regions mentioned in a typical quarterly field report, in general?")
print(basic_answer)

In a typical quarterly field report, the four regions mentioned are generally:

1. **North Region**  
2. **South Region**  
3. **East Region**  
4. **West Region**

These regions help organize data and insights geographically, making it easier to analyze performance, trends, and challenges in different areas. The exact naming or division might vary depending on the organization or industry.


In [24]:
# 2) The same kind of call, but with a system prompt that changes the style/role
terse_answer = call_llm(
    "What are the four regions mentioned in a typical quarterly field report, in general?",
    system="You are a terse, no-nonsense operations analyst. Answer in at most two short bullet points.",
)
print(terse_answer)

- North America, Europe, Asia-Pacific, Latin America  
- Sometimes Middle East & Africa included as a fourth region instead of Latin America


In [25]:
# 3) A call where you deliberately set max_output_tokens very low for a request
#    that wants a long answer. Print the text, plus response.status and
#    response.incomplete_details, so you can see what actually happened (you'll
#    likely need to call the API directly here, or extend call_llm to optionally
#    return the raw response object instead of just the text).
truncated_response = client.responses.create(
    model=MODEL_FAST,
    instructions="You are a detailed technical writer.",
    input="Write a thorough, multi-paragraph explanation of how PDF text extraction works, "
          "covering embedded text layers, font encoding, and OCR fallbacks.",
    max_output_tokens=16,
)

print("output_text:", repr(truncated_response.output_text))
print("status:", truncated_response.status)
print("incomplete_details:", truncated_response.incomplete_details)

output_text: 'PDF text extraction is a complex process that involves interpreting the internal structure of a PDF'
status: incomplete
incomplete_details: IncompleteDetails(reason='max_output_tokens')


**question 1:** What did lowering `max_output_tokens` actually do — error, truncate, or something else? 

_Reducing the `max_output_tokens` value did not cause an error, and the call still executed successfully.
However, the result was that as soon as the model reached the token limit, the output text was cut off mid-sentence.
This can be detected because `response.status` returns a value of "incomplete," with the reason specified as "max_output_tokens" in `response.incomplete_details.reason`.
The issue is that the `call_llm` function returns only the `output_text`; consequently, these fields are normally never seen, and the text truncation simply appears as a short (or incomplete) response without any warning._

## Part 2 — Traditional PDF extraction (20 pts)

In [26]:
def extract_text_by_page(pdf_path):
    """Return a list of strings, one per page, using pdfplumber's text extraction."""
    # open the PDF with pdfplumber and call .extract_text() on each page.
    # Careful: extract_text() can return None on a blank/image-only page.
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # normalize that to an empty string to keep downstream code simple.
            page_text = page.extract_text()
            texts.append(page_text if page_text else "")
    return texts


def extract_tables_by_page(pdf_path):
    """Return a list (one entry per page) of that page's extracted tables
    (each table is a list of rows, each row a list of cell strings)."""
    # use page.extract_tables()
    tables_by_page = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables_by_page.append(page.extract_tables())
    return tables_by_page


In [27]:
texts = extract_text_by_page(SAMPLE_PDF)
tables = extract_tables_by_page(SAMPLE_PDF)

for i, (t, tb) in enumerate(zip(texts, tables), start=1):
    n_chars = len(t) if t else 0
    print(f"Page {i}: {n_chars} chars, {len(tb)} table(s)")


Page 1: 783 chars, 0 table(s)
Page 2: 332 chars, 1 table(s)
Page 3: 45 chars, 0 table(s)


**question 2:** which page(s) did this handle well, and which failed? What specifically about that page caused it?

_Pages 1 and 2 worked well; they contain actual, selectable text, so pdfplumber cleanly and correctly extracted the introductory paragraph and parsed the Q3 regional table (Riverbend, Lakeside, Northgate, Fairview).
Page 3 essentially failed: it contained only 45 characters and no tables were found because that page is actually a scanned image rather than real text.
pdfplumber can only read text stored as actual text within the PDF; it does not perform OCR.
Consequently, a scanned page is invisible to it, except for a small portion of the title text located outside the image area._

## Part 3 — Multimodal LLM extraction (30 pts)

### 3a — Build the pipeline yourself (15 pts)

In [28]:
def render_page_to_image(pdf_path, page_number, dpi=150):
    """Render a single PDF page (1-indexed) to a PIL Image."""
    # TODO: use pdf2image.convert_from_path with first_page/last_page set to
    # page_number, dpi=dpi, and return the single resulting image.
    images = convert_from_path(
        pdf_path,
        dpi=dpi,
        first_page=page_number,
        last_page=page_number,
        poppler_path=r"C:\poppler\Library\bin",
    )
    return images[0]


def image_to_base64(img, fmt="PNG"):
    """Given helper: encode a PIL image as a base64 string."""
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")


def extract_with_vision(image, model=MODEL_VISION, media_type="image/png"):
    """Send a page image to a vision-capable model and return its structured
    transcription of the page (ask for the text content and note any tables)."""
    # TODO: build a client.responses.create(...) call whose input content list
    # has an "input_image" item (image_url=f"data:{media_type};base64,{...}",
    # using image_to_base64(image)) followed by an "input_text" item with your
    # extraction prompt. Return response.output_text.
    EXTRACTION_PROMPT = (
        "Extract all text and table content visible in this image of a document page. "
        "Preserve the original structure: return normal paragraphs as plain text, "
        "and return any tables as markdown tables with all rows and columns intact. "
        "If the page contains handwritten or scanned content, transcribe it as accurately as you can. "
        "Return ONLY the extracted content, with no introductory or explanatory text."
    )
    
    b64 = image_to_base64(image)
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:{media_type};base64,{b64}",
                    },
                    {
                        "type": "input_text",
                        "text": EXTRACTION_PROMPT,
                    },
                ],
            }
        ],
        max_output_tokens=1024,
    )
    return response.output_text


In [29]:
vision_results = {}
for page_num in range(1, len(texts) + 1):
    img = render_page_to_image(SAMPLE_PDF, page_num)
    vision_results[page_num] = extract_with_vision(img)
    print(f"--- Page {page_num} (vision) ---")
    print(vision_results[page_num][:400])
    print()


--- Page 1 (vision) ---
GreenLeaf Analytics

Q3 Regional Field Report

This report summarizes GreenLeaf Analytics’ third-quarter field operations across our four regional distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the Riverbend and Lakeside facilities. Staffing levels remained steady, though two sites reported temporary slowdowns tied to equipment maintenance in Aug

--- Page 2 (vision) ---
Regional Performance - Q3

| Region     | Q1 Units | Q2 Units | Q3 Units | QoQ Growth |
|------------|----------|----------|----------|------------|
| Riverbend  | 12,400   | 13,100   | 15,850   | +21.0%     |
| Lakeside   | 9,800    | 10,050   | 11,760   | +17.0%     |
| Northgate  | 14,200   | 13,900   | 13,650   | -1.8%      |
| Fairview   | 7,300    | 7,410    | 7,955    | +7.4%      |

Northg

--- Page 3 (vision) ---
Appendix A: Field Memo (scanned, as received)

______________________________________________________

GREENLEAF ANALYTICS -

### 3b — Use native PDF support (15 pts)

OpenAI's vision-capable models can also accept a PDF directly as an `input_file`
content item, instead of you rendering pages to images yourself — the API extracts
both the text and page images internally. 

In [30]:
NATIVE_PDF_PROMPT = (
    "Extract all text and table content from this PDF document, page by page. "
    "Label each page clearly, e.g. '--- Page 1 ---', before its content. "
    "Preserve the original structure: return normal paragraphs as plain text, "
    "and return any tables as markdown tables with all rows and columns intact. "
    "If a page contains scanned or handwritten content, transcribe it as accurately "
    "as possible, and mark any word or phrase you cannot confidently read as [illegible]. "
    "Return ONLY the extracted content, with no introductory or explanatory text, "
    "no summaries, and no commentary."
)

def extract_with_native_pdf(pdf_path, model=MODEL_VISION, prompt=None):
    """Send the whole PDF to the model using native PDF support and return its
    response text."""
    # TODO: base64-encode the PDF file's bytes, build a client.responses.create(...)
    # call whose input content list has an "input_file" item (filename=..., and
    # file_data=f"data:application/pdf;base64,{...}") followed by an "input_text"
    # item with your extraction prompt. Return response.output_text.
    if prompt is None:
        prompt = NATIVE_PDF_PROMPT

    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    b64_pdf = base64.standard_b64encode(pdf_bytes).decode("utf-8")

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": os.path.basename(pdf_path),
                        "file_data": f"data:application/pdf;base64,{b64_pdf}",
                    },
                    {
                        "type": "input_text",
                        "text": prompt,
                    },
                ],
            }
        ],
        max_output_tokens=2048,
    )
    return response.output_text


In [31]:
native_result = extract_with_native_pdf(SAMPLE_PDF)
print(native_result)


--- Page 1 ---

GreenLeaf Analytics  
Q3 Regional Field Report  
This report summarizes GreenLeaf Analytics' third-quarter field operations across our four regional distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the Riverbend and Lakeside facilities. Staffing levels remained steady, though two sites reported temporary slowdowns tied to equipment maintenance in August. The sections below break down performance by region and flag any open items that require follow-up before the Q4 planning cycle begins.  
Section 2 (next page) presents the regional sales and throughput table referenced above. Section 3 is a scanned field memo from the Riverbend site supervisor, included as-received, describing an inventory discrepancy that is still under review.

--- Page 2 ---

Regional Performance - Q3

| Region    | Q1 Units | Q2 Units | Q3 Units | QoQ Growth |
|-----------|----------|----------|----------|------------|
| Riverbend | 12,400   | 13,10

## Part 4 — Compare and analyze (25 pts)

In [32]:
# TODO: build a small DataFrame comparing, for each page: traditional char count,
# traditional table count, vision output length, and a short free-text "notes" column
# (e.g. "traditional got nothing; vision fully read the memo").
comparison_rows = []
for i in range(1, len(texts) + 1):
    trad_chars = len(texts[i - 1]) if texts[i - 1] else 0
    trad_tables = len(tables[i - 1])
    vision_len = len(vision_results.get(i, ""))

    if i == 1:
        notes = "Clean text layer; traditional parser fully sufficient, vision call is redundant here."
    elif i == 2:
        notes = "Real ruled table; traditional parser extracted it correctly, vision reproduces it as Markdown but costs an API call for no extra accuracy."
    else:
        notes = "Scanned image page; traditional parser got almost nothing (just the caption), vision model reads the memo directly off the pixels."

    comparison_rows.append({
        "page": i,
        "traditional_chars": trad_chars,
        "traditional_tables": trad_tables,
        "vision_output_len": vision_len,
        "notes": notes,
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,page,traditional_chars,traditional_tables,vision_output_len,notes
0,1,783,0,786,Clean text layer; traditional parser fully suf...
1,2,332,1,511,Real ruled table; traditional parser extracted...
2,3,45,0,613,Scanned image page; traditional parser got alm...


**Reflection (answer all four):**

1. Which method produced the most useful output for each page, and why?

   _For Page 1 (plain text) and Page 2 (table), the traditional method also performed well with the vision model; thus, invoking the model was unnecessary. Only the vision model correctly read Page 3 (scanned)—extracting 613 characters compared to the traditional method's 45—because pdfplumber lacks OCR capabilities._

2. 3a and 3b both use the model's vision capability under the hood. What's the practical
   trade-off between building the image pipeline yourself vs. using native PDF support?

   _3a (manual rendering) offers greater control but requires more code and one API call per page. 3b (native PDF) transmits the entire file in a single call; it is simpler, but you have less control over the rendering of each page._

3. If you were processing thousands of PDFs a day, how would you decide when it's worth
   paying for an LLM call vs. using the free traditional parser? Sketch a simple decision rule.

   _Always run the free parser first; only resort to the vision model if the page text is shorter than a certain threshold (e.g., 100 characters). This follows the same logic as hybrid_extract._

4. Name one kind of page/document you'd expect **all three** methods to struggle with. Why?

   _A scanned page with faint or blurry handwriting. A traditional parser captures nothing at all, as it lacks OCR capabilities; meanwhile, Vision and native PDF methods—if the image is so indistinct that the model itself is uncertain—will either make errors or return (illegible) results._


## a hybrid pipeline

Implement a hybrid pipeline: use the traditional parser by default, and only fall back to
the (more expensive) vision pipeline when the extracted text for a page looks too short
or empty to be real.

In [33]:
#answered
def hybrid_extract(pdf_path, min_chars=100):
    """Return a list of per-page text: traditional extraction where it looks
    sufficient, vision extraction as a fallback otherwise."""
    page_texts = extract_text_by_page(pdf_path)
    results = []
    for i, text in enumerate(page_texts, start=1):
        if text and len(text.strip()) >= min_chars:
            results.append(text)
        else:
            img = render_page_to_image(pdf_path, i)
            results.append(extract_with_vision(img))
    return results


In [34]:
hybrid_results = hybrid_extract(SAMPLE_PDF)
for i, r in enumerate(hybrid_results, start=1):
    print(f"--- Page {i} (hybrid, {len(r)} chars) ---")
    print(r[:300])
    print()

--- Page 1 (hybrid, 783 chars) ---
GreenLeaf Analytics
Q3 Regional Field Report
This report summarizes GreenLeaf Analytics' third-quarter field operations across our four regional
distribution hubs. Overall throughput increased relative to Q2, driven largely by the expansion of the
Riverbend and Lakeside facilities. Staffing levels r

--- Page 2 (hybrid, 332 chars) ---
Regional Performance - Q3
Region Q1 Units Q2 Units Q3 Units QoQ Growth
Riverbend 12,400 13,100 15,850 +21.0%
Lakeside 9,800 10,050 11,760 +17.0%
Northgate 14,200 13,900 13,650 -1.8%
Fairview 7,300 7,410 7,955 +7.4%
Northgate is the only region with negative QoQ growth this quarter; see the open main

--- Page 3 (hybrid, 616 chars) ---
Appendix A: Field Memo (scanned, as received)

__________________________________________________________

GREENLEAF ANALYTICS - FIELD MEMO

Date: 14 March   |  Site: Riverbend Warehouse   |  Ref: FM-0192

Inventory discrepancy noted during the Tuesday audit.
Pallets logged: 214  Pallets co

### This output demonstrates that `hybrid_extract` keeps pages with sufficient text free and fast to process, sending only the scanned page that actually requires it to the costlier model.In other words, out of three pages, only a single API call was consumed.